In [1]:
from pathlib import Path
import pandas as pd
import math
from faster_whisper import WhisperModel
from IPython.display import Audio, display

In [2]:
BASE_DIR = Path(r"D:\AI\speech_dataset")
METADATA_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv"
AUDIO_DIR = BASE_DIR / "processed_audio"
OUTPUT_PATH = BASE_DIR / "large_v3_stt_results.csv"

df = pd.read_csv(METADATA_PATH, sep="|")

print("Total records:", len(df))
print("Columns:", df.columns.tolist())
print("Metadata loaded successfully")

Total records: 8566
Columns: ['audio_file', 'text', 'text_normalized', 'speaker_name']
Metadata loaded successfully


In [1]:
from faster_whisper import WhisperModel

large_model = WhisperModel(
    "large-v3",
    device="cpu",
    compute_type="int8"
)

print("Large-v3 loaded successfully")

Large-v3 loaded successfully


In [5]:

from pathlib import Path

BASE_DIR = Path(r"D:\AI\speech_dataset")
METADATA_DIR = BASE_DIR / "XTTSv2_handoff"

train_path = METADATA_DIR / "metadata_train.csv"
eval_path = METADATA_DIR / "metadata_eval.csv"

for path in [train_path, eval_path]:
    print(f"\n--- {path.name} ---")

    with open(path, "r", encoding="utf-8-sig") as f:
        for line_number, line in enumerate(f, start=1):
            if line_number > 5:
                break
            print(repr(line[:300]))



--- metadata_train.csv ---
'audio_file|text|text_normalized|speaker_name\n'
'wavs/general_1_00041.wav|تماما وبجانب درجة الجودة العامة بيسألهم أسئلة تانية محددة زي إيه؟ زي تقييم مدى طبيعية الصوت هل هو متضفق ولا متقطع وتقييم صحة اللهجة هل مخارج الحروف|تماما وبجانب درجة الجودة العامة بيسألهم أسئلة تانية محددة زي إيه؟ زي تقييم مدى طبيعية الصوت هل هو متضفق ولا متقطع وتقييم صحة ال'
'wavs/medical_8_00009.wav|لأن العواقب أسوأ بكتير من اللي بنتخيله بص، اللي بيحصل لما حد يضغط على الحباية أو يفعصها زي ما التقرير بيقول هو إنه مش بس بيطلع اللي جواها البرة، لأ وموال إيه؟ في كارثة بتحصل تحت الجلد|لأن العواقب أسوأ بكتير من اللي بنتخيله بص، اللي بيحصل لما حد يضغط على الحباية أو يفعصها زي '
'wavs/sales_15_00036.wav|بنبص بره على العميل و القيمة اللي بيحصل عليها. يعني بنفكر هو مستفيد قد ايه. بالزبط بنسأل نفسنا المشكلة اللي بنحلها للعميل دي كبيرة قد إيه؟ بنوفر عليه وقت أو فلوس قد إيه؟|بنبص بره على العميل و القيمة اللي بيحصل عليها. يعني بنفكر هو مستفيد قد ايه. بالزبط بنسأل نفسنا المشكلة الل'
'wavs/sales_3_

In [8]:

from pathlib import Path
import pandas as pd

BASE_DIR = Path(r"D:\AI\speech_dataset")
METADATA_DIR = BASE_DIR / "XTTSv2_handoff"

train_path = METADATA_DIR / "metadata_train.csv"
eval_path = METADATA_DIR / "metadata_eval.csv"

# The metadata files use | as the separator
train_df = pd.read_csv(
    train_path,
    sep="|",
    encoding="utf-8-sig"
)

eval_df = pd.read_csv(
    eval_path,
    sep="|",
    encoding="utf-8-sig"
)

all_metadata = pd.concat(
    [train_df, eval_df],
    ignore_index=True
)

all_metadata = all_metadata.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

print("Training records:", len(train_df))
print("Evaluation records:", len(eval_df))
print("Total unique records:", len(all_metadata))
print("Columns:", all_metadata.columns.tolist())
print(all_metadata.head(3))


Training records: 8566
Evaluation records: 949
Total unique records: 9515
Columns: ['audio_file', 'text', 'text_normalized', 'speaker_name']
                 audio_file  \
0  wavs/general_1_00041.wav   
1  wavs/medical_8_00009.wav   
2   wavs/sales_15_00036.wav   

                                                text  \
0  تماما وبجانب درجة الجودة العامة بيسألهم أسئلة ...   
1  لأن العواقب أسوأ بكتير من اللي بنتخيله بص، الل...   
2  بنبص بره على العميل و القيمة اللي بيحصل عليها....   

                                     text_normalized speaker_name  
0  تماما وبجانب درجة الجودة العامة بيسألهم أسئلة ...   SPEAKER_02  
1  لأن العواقب أسوأ بكتير من اللي بنتخيله بص، الل...   SPEAKER_02  
2  بنبص بره على العميل و القيمة اللي بيحصل عليها....   SPEAKER_01  


In [ ]:

from pathlib import Path
import pandas as pd
import time
import math

# ==========================================
# 1. Paths and settings
# ==========================================
BASE_DIR = Path(r"D:\AI\speech_dataset")
AUDIO_DIR = BASE_DIR / "processed_audio"
OUTPUT_PATH = BASE_DIR / "large_v3_stt_results.csv"

BATCH_SIZE = 2000
SAVE_EVERY = 10

# ==========================================
# 2. Prepare all metadata
# ==========================================
data = all_metadata.dropna(
    subset=["audio_file", "text"]
).copy()

data["audio_file"] = data["audio_file"].astype(str)
data = data.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

total = len(data)

print("Total metadata records:", total)
print("Batch size:", BATCH_SIZE)
print("Number of batches:", math.ceil(total / BATCH_SIZE))

# ==========================================
# 3. Load previous results safely
# ==========================================
if OUTPUT_PATH.exists():
    old_results = pd.read_csv(OUTPUT_PATH)

    if "audio_file" not in old_results.columns:
        raise ValueError(
            "The existing CSV has no audio_file column. "
            "Rename the file before continuing."
        )

    # Only rows explicitly marked completed are skipped.
    # Old files without status are treated as needing review.
    if "status" not in old_results.columns:
        old_results["status"] = "needs_review"

    results_by_file = {}

    for row in old_results.to_dict("records"):
        filename = row.get("audio_file")

        if pd.notna(filename):
            results_by_file[str(filename)] = row

    completed_files = {
        filename
        for filename, result in results_by_file.items()
        if result.get("status") == "completed"
    }

    print("Previous result rows:", len(results_by_file))
    print("Previously completed:", len(completed_files))

else:
    results_by_file = {}
    completed_files = set()

    print("No previous CSV found. Starting fresh.")

# ==========================================
# 4. Process records in batches
# ==========================================
run_start = time.perf_counter()
processed_this_run = 0
errors_this_run = 0


def save_results():
    """Save all accumulated results to the same CSV."""
    if results_by_file:
        pd.DataFrame(
            list(results_by_file.values())
        ).to_csv(
            OUTPUT_PATH,
            index=False,
            encoding="utf-8-sig"
        )


try:
    for batch_start in range(0, total, BATCH_SIZE):

        batch = data.iloc[
            batch_start:batch_start + BATCH_SIZE
        ]

        batch_number = batch_start // BATCH_SIZE + 1

        print(
            f"\n===== Batch {batch_number} "
            f"({batch_start + 1}-{batch_start + len(batch)}) ====="
        )

        for _, row in batch.iterrows():

            audio_name = row["audio_file"]

            # Skip only successful recordings
            if audio_name in completed_files:
                continue

            original_text = str(row["text"])
            audio_path = AUDIO_DIR / audio_name

            result = {
                "audio_file": audio_name,
                "original_text": original_text,
                "predicted_text": "",
                "processing_time_seconds": None,
                "confidence_score": None,
                "status": "",
                "error_message": ""
            }

            try:
                if not audio_path.is_file():
                    result["status"] = "missing_audio"
                    result["error_message"] = (
                        f"Audio file not found: {audio_path}"
                    )

                else:
                    start = time.perf_counter()

                    segments, info = large_model.transcribe(
                        str(audio_path),
                        language="ar",
                        beam_size=1,
                        vad_filter=True
                    )

                    # Transcription runs as segments are consumed
                    segments = list(segments)

                    elapsed = time.perf_counter() - start

                    predicted_text = " ".join(
                        segment.text.strip()
                        for segment in segments
                    ).strip()

                    logprobs = [
                        segment.avg_logprob
                        for segment in segments
                        if segment.text.strip()
                    ]

                    confidence = (
                        math.exp(
                            sum(logprobs) / len(logprobs)
                        )
                        if logprobs else None
                    )

                    result["predicted_text"] = predicted_text
                    result["processing_time_seconds"] = round(
                        elapsed, 2
                    )
                    result["confidence_score"] = (
                        round(confidence, 4)
                        if confidence is not None
                        else None
                    )
                    result["status"] = "completed"

            except Exception as e:
                result["status"] = "error"
                result["error_message"] = str(e)
                errors_this_run += 1

                print(f"Error processing {audio_name}: {e}")

            # Keep the newest result for this audio file
            results_by_file[audio_name] = result

            if result["status"] == "completed":
                completed_files.add(audio_name)

            processed_this_run += 1

            # Save periodically instead of rewriting the CSV
            # after every single recording
            if processed_this_run % SAVE_EVERY == 0:
                save_results()

                print(
                    f"Processed this run: {processed_this_run} | "
                    f"Completed: {len(completed_files)}/{total}"
                )

        # Save at the end of each batch
        save_results()
        print(f"Batch {batch_number} saved successfully.")

except KeyboardInterrupt:
    # Preserve current progress if execution is interrupted
    save_results()

    print("\nExecution interrupted.")
    print("Progress saved to:", OUTPUT_PATH)
    print("Run the processing cell again to resume.")

# ==========================================
# 5. Final save and summary
# ==========================================
finally:
    save_results()

final_df = pd.DataFrame(
    list(results_by_file.values())
)

print("\n===== PROCESSING SUMMARY =====")
print("Output file:", OUTPUT_PATH)
print("Metadata records:", total)
print("Result rows saved:", len(final_df))
print(
    "Completed:",
    (final_df["status"] == "completed").sum()
)
print(
    "Errors:",
    (final_df["status"] == "error").sum()
)
print(
    "Missing audio:",
    (final_df["status"] == "missing_audio").sum()
)
print(
    "Elapsed minutes:",
    round((time.perf_counter() - run_start) / 60, 2)
)

print("\nStatus breakdown:")
print(final_df["status"].value_counts(dropna=False))


Total metadata records: 9515
Batch size: 2000
Number of batches: 5
No previous CSV found. Starting fresh.

===== Batch 1 (1-2000) =====
Processed this run: 10 | Completed: 10/9515
Processed this run: 20 | Completed: 20/9515
Processed this run: 30 | Completed: 30/9515
Processed this run: 40 | Completed: 40/9515
Processed this run: 50 | Completed: 50/9515
Processed this run: 60 | Completed: 60/9515
Processed this run: 70 | Completed: 70/9515
Processed this run: 80 | Completed: 80/9515
Processed this run: 90 | Completed: 90/9515
Processed this run: 100 | Completed: 100/9515
Processed this run: 110 | Completed: 110/9515


In [9]:
from pathlib import Path
import pandas as pd
import time
import math
from faster_whisper import WhisperModel

# 1) Paths
BASE_DIR = Path(r"D:\AI\speech_dataset")

AUDIO_DIR = BASE_DIR / "processed_audio"
TRAIN_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv"
EVAL_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_eval.csv"

OUTPUT_PATH = BASE_DIR / "whisper_small_test.csv"

# 2) Load metadata
train_df = pd.read_csv(
    TRAIN_PATH, sep="|", encoding="utf-8-sig"
)
eval_df = pd.read_csv(
    EVAL_PATH, sep="|", encoding="utf-8-sig"
)

all_metadata = pd.concat(
    [train_df, eval_df], ignore_index=True
)

all_metadata = all_metadata.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

# 3) Load existing results, if any
if OUTPUT_PATH.exists():
    results_df = pd.read_csv(
        OUTPUT_PATH, encoding="utf-8-sig"
    )
else:
    results_df = pd.DataFrame(columns=[
        "audio_file",
        "original_text",
        "predicted_text",
        "processing_time_seconds",
        "confidence_score",
        "status",
        "error_message"
    ])

# Remove duplicate rows from any previous runs
results_df = results_df.drop_duplicates(
    subset=["audio_file"], keep="last"
).reset_index(drop=True)

completed_files = set(
    results_df.loc[
        results_df["status"].eq("completed"), "audio_file"
    ].astype(str)
)

# 4) Select ONE unprocessed recording
remaining = all_metadata[
    ~all_metadata["audio_file"].astype(str).isin(completed_files)
]

if remaining.empty:
    print("No unprocessed recordings remain.")
else:
    sample = remaining.sample(n=1, random_state=42).iloc[0]

    audio_name = str(sample["audio_file"])
    audio_path = AUDIO_DIR / audio_name

    print("Test recording:", audio_name)
    print("Loading Whisper Small...")

    # 5) Load the small model
    small_model = WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )

    print("Model loaded. Starting transcription...")

    start_time = time.time()

    try:
        segments, info = small_model.transcribe(
            str(audio_path),
            language="ar",
            beam_size=1,
            vad_filter=True
        )

        segments = list(segments)
        predicted_text = " ".join(
            segment.text.strip() for segment in segments
        ).strip()

        processing_time = round(
            time.time() - start_time, 2
        )

        # Approximate score derived from segment log probabilities
        confidence_score = (
            round(math.exp(
                sum(s.avg_logprob for s in segments)
                / len(segments)
            ), 4)
            if segments else None
        )

        new_result = {
            "audio_file": audio_name,
            "original_text": sample["text"],
            "predicted_text": predicted_text,
            "processing_time_seconds": processing_time,
            "confidence_score": confidence_score,
            "status": "completed",
            "error_message": ""
        }

    except Exception as e:
        new_result = {
            "audio_file": audio_name,
            "original_text": sample["text"],
            "predicted_text": "",
            "processing_time_seconds": round(
                time.time() - start_time, 2
            ),
            "confidence_score": None,
            "status": "error",
            "error_message": str(e)
        }

    # 6) Save immediately without duplicating this filename
    results_df = results_df[
        results_df["audio_file"].astype(str) != audio_name
    ]

    results_df = pd.concat(
        [results_df, pd.DataFrame([new_result])],
        ignore_index=True
    )

    results_df.to_csv(
        OUTPUT_PATH, index=False, encoding="utf-8-sig"
    )

    print("\nTest finished.")
    print("Original:", new_result["original_text"])
    print("Prediction:", new_result["predicted_text"])
    print("Processing time:", processing_time if new_result["status"] == "completed" else new_result["processing_time_seconds"], "seconds")
    print("Confidence score:", new_result["confidence_score"])
    print("Status:", new_result["status"])
    print("Saved to:", OUTPUT_PATH)

Test recording: wavs/general_22_00029.wav
Loading Whisper Small...
Model loaded. Starting transcription...


KeyboardInterrupt: 

In [ ]:

from pathlib import Path
import pandas as pd
import time
import math
from faster_whisper import WhisperModel
from IPython.display import display, Audio, Markdown

# 1. Paths
BASE_DIR = Path(r"D:\AI\speech_dataset")
AUDIO_DIR = BASE_DIR / "processed_audio"
TRAIN_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv"
EVAL_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_eval.csv"

# New file for this first test
OUTPUT_PATH = BASE_DIR / "whisper_small_first_10.csv"

# 2. Load metadata
train_df = pd.read_csv(TRAIN_PATH, sep="|", encoding="utf-8-sig")
eval_df = pd.read_csv(EVAL_PATH, sep="|", encoding="utf-8-sig")

all_metadata = pd.concat(
    [train_df, eval_df], ignore_index=True
)
all_metadata = all_metadata.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

# 3. Select 10 random recordings
samples = all_metadata.sample(
    n=min(10, len(all_metadata)), random_state=42
)

print(f"Selected {len(samples)} recordings.")
print("Loading Whisper Small...")

# 4. Load the model once
model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print("Model loaded. Starting test...\n")

# 5. Transcribe, display, and save each recording
results = []

for i, (_, row) in enumerate(samples.iterrows(), start=1):
    audio_name = str(row["audio_file"])

    # Metadata path contains "wavs/filename.wav"
    audio_path = AUDIO_DIR / audio_name

    display(Markdown(f"## Recording {i}/{len(samples)}"))
    display(Markdown(f"**File:** `{audio_name}`"))

    original_text = str(row["text"])
    display(Markdown(f"**Original Text:** {original_text}"))

    if not audio_path.exists():
        print(f"Audio not found at: {audio_path}")
        results.append({
            "audio_file": audio_name,
            "original_text": original_text,
            "predicted_text": "",
            "processing_time_seconds": None,
            "confidence_score": None,
            "status": "error",
            "error_message": f"File not found: {audio_path}"
        })
        pd.DataFrame(results).to_csv(
            OUTPUT_PATH, index=False, encoding="utf-8-sig"
        )
        print("-" * 50)
        continue

    start_time = time.time()

    try:
        segments, info = model.transcribe(
            str(audio_path),
            language="ar",
            beam_size=1,
            vad_filter=True
        )

        segments = list(segments)
        predicted_text = " ".join(
            segment.text.strip() for segment in segments
        ).strip()

        elapsed = round(time.time() - start_time, 2)

        confidence_score = (
            round(math.exp(
                sum(s.avg_logprob for s in segments) / len(segments)
            ), 4)
            if segments else None
        )

        result = {
            "audio_file": audio_name,
            "original_text": original_text,
            "predicted_text": predicted_text,
            "processing_time_seconds": elapsed,
            "confidence_score": confidence_score,
            "status": "completed",
            "error_message": ""
        }

        display(Markdown(
            f"**Whisper Small Prediction:** "
            f"{predicted_text or '(No speech detected)'}"
        ))

        display(Markdown("**Original Audio — Play Below:**"))
        display(Audio(filename=str(audio_path)))

        print(f"Processing time: {elapsed} seconds")

    except Exception as e:
        result = {
            "audio_file": audio_name,
            "original_text": original_text,
            "predicted_text": "",
            "processing_time_seconds": round(
                time.time() - start_time, 2
            ),
            "confidence_score": None,
            "status": "error",
            "error_message": repr(e)
        }
        print("Error:", repr(e))

    results.append(result)

    # Save immediately after every recording
    pd.DataFrame(results).to_csv(
        OUTPUT_PATH, index=False, encoding="utf-8-sig"
    )

    print("Status:", result["status"])
    print("-" * 50)

print("\nTest finished.")
print("Results saved to:", OUTPUT_PATH)


Selected 10 recordings.
Loading Whisper Small...
Model loaded. Starting test...



## Recording 1/10

**File:** `wavs/general_22_00029.wav`

**Original Text:** عائد الوديعة ما بقاش مكفي محتاجين ندفع كلنا فلوس زيادة اسمها فروق صيانة لا لا لا يعني الملك بيدفع مرتين مرة وديعة بالملايين ومرة تانية عشان يصلح الإهمال اللي حصل بفلوس الوديعة أصلا طب إيه الحل؟

**Whisper Small Prediction:** عائد الوديع عما بقاش مكفي محتاجين ندفع كلنا فلوس زيادة اسمها فروق صيانة لا لا لا يعني الملك بيتفع مرتين مرة وديعة بالملايين ومرتانية عشان يصلح الإهمال اللي حصل بفلوس الوديع أصلا طبق الحل

**Original Audio — Play Below:**

... (51 outputs left out)

**Original Text:** لازم يسيب مسافات صغيرة جداً بين البلاط عشان يتمدد لو لزقهم في بعض أي تمدد بيخليه يضغط على بعضه ويتقطق طب إيه الحل العملي اللي بيقوله التقرير؟ الحل زكي جداً

**Whisper Small Prediction:** لازم يسيب مسافات صغيرة جداً بين البلاط عشان يتمدد لو لزاهم في بعض أي تمدد بيخلي يضغط على بعضه ويطأطأ طب اي الحل العملي اللي بيقوله التقرير؟ الحل زكي جداً

**Original Audio — Play Below:**

Processing time: 9.13 seconds
Status: completed
--------------------------------------------------

Test finished.
Results saved to: D:\AI\speech_dataset\whisper_small_first_10.csv


In [10]:

import pandas as pd
import time
import math
from pathlib import Path
from faster_whisper import WhisperModel

# =========================
# 1. Paths
# =========================
BASE_DIR = Path(r"D:\AI\speech_dataset")

TRAIN_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv"
EVAL_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_eval.csv"
AUDIO_DIR = BASE_DIR / "processed_audio"

LARGE_RESULTS = BASE_DIR / "large_v3_stt_results.csv"
SMALL_RESULTS = BASE_DIR / "whisper_small_continued.csv"

# ابدئي بـ 10 كتجربة، وبعدها خليها None لمعالجة الباقي
MAX_NEW_FILES = None

# =========================
# 2. Load metadata
# =========================
train_df = pd.read_csv(
    TRAIN_PATH, sep="|", encoding="utf-8-sig"
)
eval_df = pd.read_csv(
    EVAL_PATH, sep="|", encoding="utf-8-sig"
)

all_metadata = pd.concat(
    [train_df, eval_df], ignore_index=True
)

all_metadata = all_metadata.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

# =========================
# 3. Load previous results
# =========================
large_df = pd.read_csv(
    LARGE_RESULTS, encoding="utf-8-sig"
)

if SMALL_RESULTS.exists():
    small_df = pd.read_csv(
        SMALL_RESULTS, encoding="utf-8-sig"
    )
else:
    small_df = pd.DataFrame(columns=[
        "audio_file",
        "original_text",
        "predicted_text",
        "processing_time_seconds",
        "confidence_score",
        "status",
        "error_message"
    ])

# استثناء كل الملفات التي حاول Large-v3 معالجتها
large_files = set(
    large_df["audio_file"].dropna().astype(str)
)

# استثناء الملفات التي سبق تسجيلها في نتائج Small
small_files = set(
    small_df["audio_file"].dropna().astype(str)
)

# =========================
# 4. Select remaining files
# =========================
remaining = all_metadata[
    ~all_metadata["audio_file"].astype(str).isin(large_files)
    & ~all_metadata["audio_file"].astype(str).isin(small_files)
].copy()

if MAX_NEW_FILES is not None:
    remaining = remaining.head(MAX_NEW_FILES)

print("Total metadata records:", len(all_metadata))
print("Large-v3 records:", len(large_files))
print("Previously saved Small records:", len(small_files))
print("Selected for this run:", len(remaining))

if remaining.empty:
    print("No new files to process.")
else:
    # =========================
    # 5. Load Whisper Small
    # =========================
    print("\nLoading Whisper Small...")
    model = WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )
    print("Model loaded. Starting transcription...\n")

    # =========================
    # 6. Process recordings
    # =========================
    for index, (_, row) in enumerate(
        remaining.iterrows(), start=1
    ):
        audio_name = str(row["audio_file"])
        audio_path = AUDIO_DIR / audio_name

        original_text = str(row["text"])
        start_time = time.time()

        result = {
            "audio_file": audio_name,
            "original_text": original_text,
            "predicted_text": "",
            "processing_time_seconds": 0,
            "confidence_score": None,
            "status": "error",
            "error_message": ""
        }

        try:
            if not audio_path.exists():
                raise FileNotFoundError(
                    f"Audio not found: {audio_path}"
                )

            segments, info = model.transcribe(
                str(audio_path),
                language="ar",
                beam_size=1,
                vad_filter=True
            )

            segments = list(segments)
            predicted_text = " ".join(
                segment.text.strip()
                for segment in segments
            ).strip()

            # مؤشر تقريبي للثقة، وليس احتمالًا معايرًا
            logprobs = [
                segment.avg_logprob
                for segment in segments
                if segment.avg_logprob is not None
            ]

            if logprobs:
                mean_logprob = sum(logprobs) / len(logprobs)
                confidence = round(
                    math.exp(mean_logprob), 4
                )
            else:
                confidence = None

            result["predicted_text"] = predicted_text
            result["confidence_score"] = confidence
            result["status"] = "completed"

        except Exception as e:
            result["error_message"] = str(e)

        result["processing_time_seconds"] = round(
            time.time() - start_time, 2
        )

        # حفظ النتيجة بعد كل ملف حتى لا نخسر التقدم
        small_df = pd.concat(
            [small_df, pd.DataFrame([result])],
            ignore_index=True
        )
        small_df.to_csv(
            SMALL_RESULTS,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"[{index}/{len(remaining)}] {audio_name} | "
            f"Status: {result['status']} | "
            f"Time: {result['processing_time_seconds']}s | "
            f"Confidence: {result['confidence_score']}"
        )

    print("\nFinished this run.")
    print("Results saved to:", SMALL_RESULTS)


Total metadata records: 9515
Large-v3 records: 110
Previously saved Small records: 18
Selected for this run: 9387

Loading Whisper Small...
Model loaded. Starting transcription...

[1/9387] wavs/sales_23_00085.wav | Status: completed | Time: 9.73s | Confidence: 0.8057
[2/9387] wavs/sales_45_00067.wav | Status: completed | Time: 8.88s | Confidence: 0.778
[3/9387] wavs/general_47_00023.wav | Status: completed | Time: 9.4s | Confidence: 0.7485
[4/9387] wavs/sales_7_00035.wav | Status: completed | Time: 10.29s | Confidence: 0.7604
[5/9387] wavs/general_17_00005.wav | Status: completed | Time: 9.08s | Confidence: 0.7028
[6/9387] wavs/general_13_00019.wav | Status: completed | Time: 11.49s | Confidence: 0.7746
[7/9387] wavs/medical_7_00013.wav | Status: completed | Time: 10.53s | Confidence: 0.8212
[8/9387] wavs/general_29_00016.wav | Status: completed | Time: 11.24s | Confidence: 0.753
[9/9387] wavs/sales_16_00084.wav | Status: completed | Time: 10.98s | Confidence: 0.7564
[10/9387] wavs/sa

KeyboardInterrupt: 

In [ ]:

import pandas as pd
import time
import math
from pathlib import Path
from faster_whisper import WhisperModel

# =========================
# 1. Paths
# =========================
BASE_DIR = Path(r"D:\AI\speech_dataset")

TRAIN_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv"
EVAL_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_eval.csv"
AUDIO_DIR = BASE_DIR / "processed_audio"

LARGE_RESULTS = BASE_DIR / "large_v3_stt_results.csv"
SMALL_RESULTS = BASE_DIR / "whisper_small_continued.csv"

# ابدئي بـ 10 كتجربة، وبعدها خليها None لمعالجة الباقي
MAX_NEW_FILES = None

# =========================
# 2. Load metadata
# =========================
train_df = pd.read_csv(
    TRAIN_PATH, sep="|", encoding="utf-8-sig"
)
eval_df = pd.read_csv(
    EVAL_PATH, sep="|", encoding="utf-8-sig"
)

all_metadata = pd.concat(
    [train_df, eval_df], ignore_index=True
)

all_metadata = all_metadata.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

# =========================
# 3. Load previous results
# =========================
large_df = pd.read_csv(
    LARGE_RESULTS, encoding="utf-8-sig"
)

if SMALL_RESULTS.exists():
    small_df = pd.read_csv(
        SMALL_RESULTS, encoding="utf-8-sig"z
    )
else:
    small_df = pd.DataFrame(columns=[
        "audio_file",
        "original_text",
        "predicted_text",
        "processing_time_seconds",
        "confidence_score",
        "status",
        "error_message"
    ])

# استثناء كل الملفات التي حاول Large-v3 معالجتها
large_files = set(
    large_df["audio_file"].dropna().astype(str)
)

# استثناء الملفات التي سبق تسجيلها في نتائج Small
small_files = set(
    small_df["audio_file"].dropna().astype(str)
)

# =========================
# 4. Select remaining files
# =========================
remaining = all_metadata[
    ~all_metadata["audio_file"].astype(str).isin(large_files)
    & ~all_metadata["audio_file"].astype(str).isin(small_files)
].copy()

if MAX_NEW_FILES is not None:
    remaining = remaining.head(MAX_NEW_FILES)

print("Total metadata records:", len(all_metadata))
print("Large-v3 records:", len(large_files))
print("Previously saved Small records:", len(small_files))
print("Selected for this run:", len(remaining))

if remaining.empty:
    print("No new files to process.")
else:
    # =========================
    # 5. Load Whisper Small
    # =========================
    print("\nLoading Whisper Small...")
    model = WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )
    print("Model loaded. Starting transcription...\n")

    # =========================
    # 6. Process recordings
    # =========================
    for index, (_, row) in enumerate(
        remaining.iterrows(), start=1
    ):
        audio_name = str(row["audio_file"])
        audio_path = AUDIO_DIR / audio_name

        original_text = str(row["text"])
        start_time = time.time()

        result = {
            "audio_file": audio_name,
            "original_text": original_text,
            "predicted_text": "",
            "processing_time_seconds": 0,
            "confidence_score": None,
            "status": "error",
            "error_message": ""
        }

        try:
            if not audio_path.exists():
                raise FileNotFoundError(
                    f"Audio not found: {audio_path}"
                )

            segments, info = model.transcribe(
                str(audio_path),
                language="ar",
                beam_size=1,
                vad_filter=True
            )

            segments = list(segments)
            predicted_text = " ".join(
                segment.text.strip()
                for segment in segments
            ).strip()

            # مؤشر تقريبي للثقة، وليس احتمالًا معايرًا
            logprobs = [
                segment.avg_logprob
                for segment in segments
                if segment.avg_logprob is not None
            ]

            if logprobs:
                mean_logprob = sum(logprobs) / len(logprobs)
                confidence = round(
                    math.exp(mean_logprob), 4
                )
            else:
                confidence = None

            result["predicted_text"] = predicted_text
            result["confidence_score"] = confidence
            result["status"] = "completed"

        except Exception as e:
            result["error_message"] = str(e)

        result["processing_time_seconds"] = round(
            time.time() - start_time, 2
        )

        # حفظ النتيجة بعد كل ملف حتى لا نخسر التقدم
        small_df = pd.concat(
            [small_df, pd.DataFrame([result])],
            ignore_index=True
        )
        small_df.to_csv(
            SMALL_RESULTS,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"[{index}/{len(remaining)}] {audio_name} | "
            f"Status: {result['status']} | "
            f"Time: {result['processing_time_seconds']}s | "
            f"Confidence: {result['confidence_score']}"
        )

    print("\nFinished this run.")
    print("Results saved to:", SMALL_RESULTS)


Total metadata records: 9515
Large-v3 records: 110
Previously saved Small records: 3255
Selected for this run: 6150

Loading Whisper Small...
Model loaded. Starting transcription...

[1/6150] wavs/sales_33_00027.wav | Status: completed | Time: 8.96s | Confidence: 0.7681
[2/6150] wavs/medical_2_00021.wav | Status: completed | Time: 8.03s | Confidence: 0.7477
[3/6150] wavs/general_47_00025.wav | Status: completed | Time: 7.95s | Confidence: 0.7958
[4/6150] wavs/general_25_00022.wav | Status: completed | Time: 7.78s | Confidence: 0.7719
[5/6150] wavs/sales_31_00059.wav | Status: completed | Time: 9.01s | Confidence: 0.7425
[6/6150] wavs/sales_12_00114.wav | Status: completed | Time: 6.8s | Confidence: 0.6435
[7/6150] wavs/general_51_00040.wav | Status: completed | Time: 7.86s | Confidence: 0.7866
[8/6150] wavs/general_12_00031.wav | Status: completed | Time: 7.7s | Confidence: 0.7812
[9/6150] wavs/sales_24_00079.wav | Status: completed | Time: 13.54s | Confidence: 0.7436
[10/6150] wavs/me

KeyboardInterrupt: 

In [3]:

import pandas as pd
import time
import math
from pathlib import Path
from faster_whisper import WhisperModel

# =========================
# 1. Paths
# =========================
BASE_DIR = Path(r"D:\AI\speech_dataset")

TRAIN_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv"
EVAL_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_eval.csv"
AUDIO_DIR = BASE_DIR / "processed_audio"

LARGE_RESULTS = BASE_DIR / "large_v3_stt_results.csv"
SMALL_RESULTS = BASE_DIR / "whisper_small_continued.csv"

# ابدئي بـ 10 كتجربة، وبعدها خليها None لمعالجة الباقي
MAX_NEW_FILES = None

# =========================
# 2. Load metadata
# =========================
train_df = pd.read_csv(
    TRAIN_PATH, sep="|", encoding="utf-8-sig"
)
eval_df = pd.read_csv(
    EVAL_PATH, sep="|", encoding="utf-8-sig"
)

all_metadata = pd.concat(
    [train_df, eval_df], ignore_index=True
)

all_metadata = all_metadata.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

# =========================
# 3. Load previous results
# =========================
large_df = pd.read_csv(
    LARGE_RESULTS, encoding="utf-8-sig"
)

if SMALL_RESULTS.exists():
    small_df = pd.read_csv(
        SMALL_RESULTS, encoding="utf-8-sig"
    )
else:
    small_df = pd.DataFrame(columns=[
        "audio_file",
        "original_text",
        "predicted_text",
        "processing_time_seconds",
        "confidence_score",
        "status",
        "error_message"
    ])

# استثناء كل الملفات التي حاول Large-v3 معالجتها
large_files = set(
    large_df["audio_file"].dropna().astype(str)
)

# استثناء الملفات التي سبق تسجيلها في نتائج Small
small_files = set(
    small_df["audio_file"].dropna().astype(str)
)

# =========================
# 4. Select remaining files
# =========================
remaining = all_metadata[
    ~all_metadata["audio_file"].astype(str).isin(large_files)
    & ~all_metadata["audio_file"].astype(str).isin(small_files)
].copy()

if MAX_NEW_FILES is not None:
    remaining = remaining.head(MAX_NEW_FILES)

print("Total metadata records:", len(all_metadata))
print("Large-v3 records:", len(large_files))
print("Previously saved Small records:", len(small_files))
print("Selected for this run:", len(remaining))

if remaining.empty:
    print("No new files to process.")
else:
    # =========================
    # 5. Load Whisper Small
    # =========================
    print("\nLoading Whisper Small...")
    model = WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )
    print("Model loaded. Starting transcription...\n")

    # =========================
    # 6. Process recordings
    # =========================
    for index, (_, row) in enumerate(
        remaining.iterrows(), start=1
    ):
        audio_name = str(row["audio_file"])
        audio_path = AUDIO_DIR / audio_name

        original_text = str(row["text"])
        start_time = time.time()

        result = {
            "audio_file": audio_name,
            "original_text": original_text,
            "predicted_text": "",
            "processing_time_seconds": 0,
            "confidence_score": None,
            "status": "error",
            "error_message": ""
        }

        try:
            if not audio_path.exists():
                raise FileNotFoundError(
                    f"Audio not found: {audio_path}"
                )

            segments, info = model.transcribe(
                str(audio_path),
                language="ar",
                beam_size=1,
                vad_filter=True
            )

            segments = list(segments)
            predicted_text = " ".join(
                segment.text.strip()
                for segment in segments
            ).strip()

            # مؤشر تقريبي للثقة، وليس احتمالًا معايرًا
            logprobs = [
                segment.avg_logprob
                for segment in segments
                if segment.avg_logprob is not None
            ]

            if logprobs:
                mean_logprob = sum(logprobs) / len(logprobs)
                confidence = round(
                    math.exp(mean_logprob), 4
                )
            else:
                confidence = None

            result["predicted_text"] = predicted_text
            result["confidence_score"] = confidence
            result["status"] = "completed"

        except Exception as e:
            result["error_message"] = str(e)

        result["processing_time_seconds"] = round(
            time.time() - start_time, 2
        )

        # حفظ النتيجة بعد كل ملف حتى لا نخسر التقدم
        small_df = pd.concat(
            [small_df, pd.DataFrame([result])],
            ignore_index=True
        )
        small_df.to_csv(
            SMALL_RESULTS,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"[{index}/{len(remaining)}] {audio_name} | "
            f"Status: {result['status']} | "
            f"Time: {result['processing_time_seconds']}s | "
            f"Confidence: {result['confidence_score']}"
        )

    print("\nFinished this run.")
    print("Results saved to:", SMALL_RESULTS)


Total metadata records: 9515
Large-v3 records: 110
Previously saved Small records: 6455
Selected for this run: 2950

Loading Whisper Small...
Model loaded. Starting transcription...

[1/2950] wavs/medical_24_00062.wav | Status: completed | Time: 10.45s | Confidence: 0.714
[2/2950] wavs/sales_2_00060.wav | Status: completed | Time: 10.97s | Confidence: 0.732
[3/2950] wavs/sales_16_00023.wav | Status: completed | Time: 8.59s | Confidence: 0.677
[4/2950] wavs/sales_29_00072.wav | Status: completed | Time: 8.35s | Confidence: 0.7125
[5/2950] wavs/medical_13_00055.wav | Status: completed | Time: 9.22s | Confidence: 0.7807
[6/2950] wavs/general_17_00003.wav | Status: completed | Time: 9.54s | Confidence: 0.7593
[7/2950] wavs/general_4_00047.wav | Status: completed | Time: 9.02s | Confidence: 0.7383
[8/2950] wavs/medical_22_00001.wav | Status: completed | Time: 8.85s | Confidence: 0.7536
[9/2950] wavs/sales_31_00008.wav | Status: completed | Time: 8.94s | Confidence: 0.6491
[10/2950] wavs/gen

KeyboardInterrupt: 

In [1]:

import pandas as pd
import time
import math
from pathlib import Path
from faster_whisper import WhisperModel

# =========================
# 1. Paths
# =========================
BASE_DIR = Path(r"D:\AI\speech_dataset")

TRAIN_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv"
EVAL_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_eval.csv"
AUDIO_DIR = BASE_DIR / "processed_audio"

LARGE_RESULTS = BASE_DIR / "large_v3_stt_results.csv"
SMALL_RESULTS = BASE_DIR / "whisper_small_continued.csv"

# ابدئي بـ 10 كتجربة، وبعدها خليها None لمعالجة الباقي
MAX_NEW_FILES = None

# =========================
# 2. Load metadata
# =========================
train_df = pd.read_csv(
    TRAIN_PATH, sep="|", encoding="utf-8-sig"
)
eval_df = pd.read_csv(
    EVAL_PATH, sep="|", encoding="utf-8-sig"
)

all_metadata = pd.concat(
    [train_df, eval_df], ignore_index=True
)

all_metadata = all_metadata.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

# =========================
# 3. Load previous results
# =========================
large_df = pd.read_csv(
    LARGE_RESULTS, encoding="utf-8-sig"
)

if SMALL_RESULTS.exists():
    small_df = pd.read_csv(
        SMALL_RESULTS, encoding="utf-8-sig"
    )
else:
    small_df = pd.DataFrame(columns=[
        "audio_file",
        "original_text",
        "predicted_text",
        "processing_time_seconds",
        "confidence_score",
        "status",
        "error_message"
    ])

# استثناء كل الملفات التي حاول Large-v3 معالجتها
large_files = set(
    large_df["audio_file"].dropna().astype(str)
)

# استثناء الملفات التي سبق تسجيلها في نتائج Small
small_files = set(
    small_df["audio_file"].dropna().astype(str)
)

# =========================
# 4. Select remaining files
# =========================
remaining = all_metadata[
    ~all_metadata["audio_file"].astype(str).isin(large_files)
    & ~all_metadata["audio_file"].astype(str).isin(small_files)
].copy()

if MAX_NEW_FILES is not None:
    remaining = remaining.head(MAX_NEW_FILES)

print("Total metadata records:", len(all_metadata))
print("Large-v3 records:", len(large_files))
print("Previously saved Small records:", len(small_files))
print("Selected for this run:", len(remaining))

if remaining.empty:
    print("No new files to process.")
else:
    # =========================
    # 5. Load Whisper Small
    # =========================
    print("\nLoading Whisper Small...")
    model = WhisperModel(
        "small",
        device="cpu",
        compute_type="int8"
    )
    print("Model loaded. Starting transcription...\n")

    # =========================
    # 6. Process recordings
    # =========================
    for index, (_, row) in enumerate(
        remaining.iterrows(), start=1
    ):
        audio_name = str(row["audio_file"])
        audio_path = AUDIO_DIR / audio_name

        original_text = str(row["text"])
        start_time = time.time()

        result = {
            "audio_file": audio_name,
            "original_text": original_text,
            "predicted_text": "",
            "processing_time_seconds": 0,
            "confidence_score": None,
            "status": "error",
            "error_message": ""
        }

        try:
            if not audio_path.exists():
                raise FileNotFoundError(
                    f"Audio not found: {audio_path}"
                )

            segments, info = model.transcribe(
                str(audio_path),
                language="ar",
                beam_size=1,
                vad_filter=True
            )

            segments = list(segments)
            predicted_text = " ".join(
                segment.text.strip()
                for segment in segments
            ).strip()

            # مؤشر تقريبي للثقة، وليس احتمالًا معايرًا
            logprobs = [
                segment.avg_logprob
                for segment in segments
                if segment.avg_logprob is not None
            ]

            if logprobs:
                mean_logprob = sum(logprobs) / len(logprobs)
                confidence = round(
                    math.exp(mean_logprob), 4
                )
            else:
                confidence = None

            result["predicted_text"] = predicted_text
            result["confidence_score"] = confidence
            result["status"] = "completed"

        except Exception as e:
            result["error_message"] = str(e)

        result["processing_time_seconds"] = round(
            time.time() - start_time, 2
        )

        # حفظ النتيجة بعد كل ملف حتى لا نخسر التقدم
        small_df = pd.concat(
            [small_df, pd.DataFrame([result])],
            ignore_index=True
        )
        small_df.to_csv(
            SMALL_RESULTS,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            f"[{index}/{len(remaining)}] {audio_name} | "
            f"Status: {result['status']} | "
            f"Time: {result['processing_time_seconds']}s | "
            f"Confidence: {result['confidence_score']}"
        )

    print("\nFinished this run.")
    print("Results saved to:", SMALL_RESULTS)


Total metadata records: 9515
Large-v3 records: 110
Previously saved Small records: 7956
Selected for this run: 1449

Loading Whisper Small...
Model loaded. Starting transcription...

[1/1449] wavs/medical_11_00011.wav | Status: completed | Time: 14.85s | Confidence: 0.7153
[2/1449] wavs/sales_1_00077.wav | Status: completed | Time: 19.77s | Confidence: 0.7185
[3/1449] wavs/sales_1_00029.wav | Status: completed | Time: 9.06s | Confidence: 0.7125
[4/1449] wavs/sales_37_00139.wav | Status: completed | Time: 9.56s | Confidence: 0.7972
[5/1449] wavs/sales_6_00104.wav | Status: completed | Time: 11.15s | Confidence: 0.7297
[6/1449] wavs/general_4_00039.wav | Status: completed | Time: 8.37s | Confidence: 0.7432
[7/1449] wavs/sales_25_00096.wav | Status: completed | Time: 8.57s | Confidence: 0.7251
[8/1449] wavs/sales_45_00043.wav | Status: completed | Time: 8.28s | Confidence: 0.7582
[9/1449] wavs/sales_23_00117.wav | Status: completed | Time: 8.33s | Confidence: 0.7009
[10/1449] wavs/sales_2

In [2]:

import pandas as pd
from pathlib import Path
from IPython.display import display, Audio, Markdown

# =========================
# 1. Paths
# =========================
BASE_DIR = Path(r"D:\AI\speech_dataset")

AUDIO_DIR = BASE_DIR / "processed_audio"

LARGE_RESULTS = BASE_DIR / "large_v3_stt_results.csv"
SMALL_RESULTS = BASE_DIR / "whisper_small_continued.csv"

# =========================
# 2. Load results
# =========================
large_df = pd.read_csv(LARGE_RESULTS, encoding="utf-8-sig")
small_df = pd.read_csv(SMALL_RESULTS, encoding="utf-8-sig")

# Combine both result files
all_results = pd.concat(
    [large_df, small_df],
    ignore_index=True
)

# Convert confidence to numeric
all_results["confidence_score"] = pd.to_numeric(
    all_results["confidence_score"],
    errors="coerce"
)

# Keep completed recordings only
all_results = all_results[
    all_results["status"] == "completed"
].copy()

# Remove duplicate audio files, if any
all_results = all_results.drop_duplicates(
    subset=["audio_file"],
    keep="last"
)

# =========================
# 3. Filter low confidence
# =========================
low_confidence = all_results[
    all_results["confidence_score"] < 0.5
].sort_values("confidence_score")

print("Total processed recordings:", len(all_results))
print("Low confidence recordings:", len(low_confidence))

# =========================
# 4. Play and display
# =========================
for i, (_, row) in enumerate(
    low_confidence.iterrows(), start=1
):

    audio_path = AUDIO_DIR / str(row["audio_file"])

    display(Markdown(f"## Recording {i}"))
    display(Markdown(
        f"**Confidence Score:** {row['confidence_score']}"
    ))

    display(Markdown(
        f"**Original Text:** {row['original_text']}"
    ))

    display(Markdown(
        f"**Predicted Text:** {row['predicted_text']}"
    ))

    if audio_path.exists():
        display(Audio(filename=str(audio_path)))
    else:
        print("Audio file not found:", audio_path)

    display(Markdown("---"))


Total processed recordings: 9515
Low confidence recordings: 14


## Recording 1

**Confidence Score:** 0.2368

**Original Text:** ال MVP هو اساس الدورة دي. و المثال اللي المصدر بيديه بيوضح الفكرة قوي. لو فكرتي موقع او ابلكيشن. ال MVP مش لازم يكون برمجة و كود من اول يوم.

**Predicted Text:** opposite to the idea our brain, the MVP is the basic of this role and the example of this source with this, will clarify the idea if you thought of a website or application MVP shouldn't be on TV for a long time

---

... (61 outputs left out)

**Confidence Score:** 0.4921

**Original Text:** ما عندهاش أوعية دموية مباشرة هي بتتغذى بعملية تشبه السفنجة لما بتتحرك الضغط بيتغير فتنتص السوائل وتطرد الفضلات الثبات بيمنع ضخ الغزة ده كأنك بتجوع غضاريفك

**Predicted Text:** يقوم بإمكانه إضافة موضوعاً لتغذب عملية تجبها السفنجة عندما تتحرك، الدخط يتغير، فتنطص السوائل وتترد الفضلات الثبات يمنع الدخل غزدة كأنك بتجوع غضاريفك

---

In [3]:

import pandas as pd
from pathlib import Path

BASE_DIR = Path(r"D:\AI\speech_dataset")

files = [
    BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv",
    BASE_DIR / "XTTSv2_handoff" / "metadata_eval.csv",
    BASE_DIR / "large_v3_stt_results.csv",
    BASE_DIR / "whisper_small_continued.csv"
]

OLD_WORD = "بالزبط"
NEW_WORD = "بالظبط"

TEXT_COLUMNS = [
    "text",
    "text_normalized",
    "original_text",
    "predicted_text"
]

for file_path in files:

    df = pd.read_csv(
        file_path,
        sep="|" if file_path.name.startswith("metadata_") else ",",
        encoding="utf-8-sig"
    )

    total_replacements = 0

    for column in TEXT_COLUMNS:
        if column in df.columns:
            count = (
                df[column]
                .fillna("")
                .astype(str)
                .str.count(OLD_WORD)
                .sum()
            )

            df[column] = df[column].apply(
                lambda x: x.replace(OLD_WORD, NEW_WORD)
                if isinstance(x, str) else x
            )

            total_replacements += count

    df.to_csv(
        file_path,
        index=False,
        sep="|" if file_path.name.startswith("metadata_") else ",",
        encoding="utf-8-sig"
    )

    print(f"{file_path.name}: {total_replacements} replacements")

print("\nAll files updated successfully!")


metadata_train.csv: 2078 replacements
metadata_eval.csv: 218 replacements
large_v3_stt_results.csv: 27 replacements
whisper_small_continued.csv: 1203 replacements

All files updated successfully!


In [4]:

import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# =========================
# 1. Paths
# =========================
BASE_DIR = Path(r"D:\AI\speech_dataset")

TRAIN_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_train.csv"
EVAL_PATH = BASE_DIR / "XTTSv2_handoff" / "metadata_eval.csv"

OUTPUT_DIR = BASE_DIR / "XTTSv2_handoff" / "final_splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# 2. Load metadata
# =========================
train_df = pd.read_csv(
    TRAIN_PATH, sep="|", encoding="utf-8-sig"
)

eval_df = pd.read_csv(
    EVAL_PATH, sep="|", encoding="utf-8-sig"
)

# Combine both datasets
all_data = pd.concat(
    [train_df, eval_df],
    ignore_index=True
)

# Remove duplicate audio records
all_data = all_data.drop_duplicates(
    subset=["audio_file"]
).reset_index(drop=True)

print("Total unique recordings:", len(all_data))

# =========================
# 3. Split: 80% Training, 20% Temporary
# =========================
stratify_col = None

# Preserve speaker distribution when possible
if (
    "speaker_name" in all_data.columns
    and all_data["speaker_name"].notna().all()
    and all_data["speaker_name"].value_counts().min() >= 3
):
    stratify_col = all_data["speaker_name"]

train_data, temp_data = train_test_split(
    all_data,
    test_size=0.20,
    random_state=42,
    stratify=stratify_col
)

# =========================
# 4. Split Temporary: 50% Validation, 50% Testing
# =========================
stratify_temp = None

if (
    "speaker_name" in temp_data.columns
    and temp_data["speaker_name"].notna().all()
    and temp_data["speaker_name"].value_counts().min() >= 2
):
    stratify_temp = temp_data["speaker_name"]

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    stratify=stratify_temp
)

# =========================
# 5. Save CSV files
# =========================
train_data.to_csv(
    OUTPUT_DIR / "metadata_train.csv",
    sep="|", index=False, encoding="utf-8-sig"
)

validation_data.to_csv(
    OUTPUT_DIR / "metadata_validation.csv",
    sep="|", index=False, encoding="utf-8-sig"
)

test_data.to_csv(
    OUTPUT_DIR / "metadata_test.csv",
    sep="|", index=False, encoding="utf-8-sig"
)

# =========================
# 6. Verify the split
# =========================
print("\n========== SPLIT RESULTS ==========")

for name, dataset in [
    ("Training", train_data),
    ("Validation", validation_data),
    ("Testing", test_data)
]:
    percentage = len(dataset) / len(all_data) * 100

    print(
        f"{name}: {len(dataset)} records "
        f"({percentage:.2f}%)"
    )

# Check for overlapping audio files
train_files = set(train_data["audio_file"])
validation_files = set(validation_data["audio_file"])
test_files = set(test_data["audio_file"])

print("\n========== DUPLICATE CHECK ==========")
print("Train & Validation:", len(train_files & validation_files))
print("Train & Test:", len(train_files & test_files))
print("Validation & Test:", len(validation_files & test_files))

print("\nSaved files in:", OUTPUT_DIR)


Total unique recordings: 9515

========== SPLIT RESULTS ==========
Training: 7612 records (80.00%)
Validation: 951 records (9.99%)
Testing: 952 records (10.01%)

========== DUPLICATE CHECK ==========
Train & Validation: 0
Train & Test: 0
Validation & Test: 0

Saved files in: D:\AI\speech_dataset\XTTSv2_handoff\final_splits
